# Prepare missing LocateAnything and SAM regions

This is a region-only GPU preparation stage. The models, prompt and proposal policy are fixed. It produces boxes and SAM masks, including explicit empty outcomes, and makes no unusual-object decision. It never loads DINO, computes P1, or opens evaluator labels.

Existing complete development regions should be imported directly into the cache preparation stage. Use this producer only for roles whose region outputs are missing, or resume its own incomplete region run.

In [ ]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = 'requirements/profiles/linux-regions-cuda126.json'
NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-native-region-inference.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    BOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)
    CODE_PROVENANCE = BOOTSTRAP['provenance']
    ROOT = CODE_ROOT
    # Scientific imports are allowed only after the exact runtime/source preflight.
    from IPython.display import display, Image
    import pandas as pd
    display({'source': CODE_PROVENANCE, 'runtime': BOOTSTRAP['runtime'],
             'preparation_only': 'Verified source/runtime does not establish protected-resource readiness'})
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _failure = {'status':'bootstrap_failed','error_type':type(_bootstrap_error).__name__,'scientific_run_started':False}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure,sort_keys=True)+'\n')
    os.replace(_temporary,_failure_root/'bootstrap-failure.json')
    raise

## Validate current gates and model inputs

Set VPI_NEXT_CONFIG. Required: role, rgb_root, output_root and current method/exposure prerequisites. RGB must be the verified canonical model-input package. All20 views are prepared because the fixed nineteen1/2/3/5-view schedules use their union; every later decision still reads only its selected views.

`action=plan` performs no inference. `action=extract` additionally requires locate_model_root with a verified immutable model_snapshot.json and sam_checkpoint. The exact Linux GPU profile must pass the setup probe first. Pin LocateAnything revision and SAM checkpoint/code revision. No automatic internet download or runtime repinning occurs.

In [ ]:
CONFIG=_runtime.resolve_config('native-regions',ROOT)
ACTION=CONFIG.get('action','plan')
from current_stage_execution import run_stage
from vpi_next_evaluation.native_regions import plan_regions,extract_regions
from current_official_region_ids import extract_regions as extract_official_regions
if ACTION not in {'plan','extract'}: raise ValueError('Choose plan or extract')
OPERATION=plan_regions if ACTION=='plan' else (extract_official_regions if CONFIG.get('role') in {'seen','unseen'} else extract_regions)
RESULT=run_stage('native-regions-'+ACTION,CONFIG,lambda:OPERATION(CONFIG,ROOT),CODE_PROVENANCE)
display(RESULT)

## Checkpoints, failure accounting and reuse

maximum_new_views bounds each invocation. The launch queue also sets shard_count/shard_index for deterministic partitions of at most100 scenes. A completed partition emits region_shard_lock.json and region_shard_manifest.json; the CPU region_merge operation verifies every partition before producing the complete role artifact_lock.json. Partial shards cannot be used as a complete cohort.

Proposal and region checkpoints are independent: a SAM failure does not rerun completed LocateAnything proposals. Two bounded LocateAnything attempts are allowed for no-valid-box, truncation or generation-timeout outcomes. OOM, permission, corrupt input and incompatible model/configuration errors stop; they are never counted as empty images. Empty proposals and masks remain explicit records.

SAM uses a bounded main-thread timeout. A stuck native GPU kernel can still require stopping the Kaggle process; retained completed checkpoints survive only if their files are exported or saved. The final artifact_lock.json seals all scene/view region records and provenance after complete coverage. Normalized mask hashes are computed from original256×256 uint8 pixels; the historical SAM hash is retained separately.

Pass region_format=current-native-regions, region_root=output_root, rgb_root and the exact artifact_lock.json SHA256 to the current cache preparation notebook. That separate stage reuses these masks and extracts only genuinely missing DINO outputs in its own CPU FP32 runtime. Do not rerun regions to repair charts or tracking.